# Классификация ботов — quickstart

Ноутбук показывает, как загрузить данные, собрать пару простейших признаков и получить
валидный `submission.csv`. Это **не** решение задачи: скор такого baseline будет чуть выше
константы. Дальше — ваша работа.

Условие и описание метрики — в `README.md`.

In [46]:
import numpy as np
import pandas as pd
import re
import math
from collections import Counter
from IPython.display import display

from feature_engineering import calculate_entropy, add_ua_features, add_temporal_features
from catboost import CatBoostClassifier

train = pd.read_csv('data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])

print(train.shape, test.shape, events.shape)
print('доля ботов в train:', train.target.mean().round(4))
train.head()

(11091, 5) (4909, 4) (328905, 14)
доля ботов в train: 0.0811


,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,0
1,ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,0
2,ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,0
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,0
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,0


In [47]:
events.info()

<class 'pandas.DataFrame'>
RangeIndex: 328905 entries, 0 to 328904
Data columns (total 14 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   cookie_id      328905 non-null  str           
 1   event_ts       328905 non-null  datetime64[us]
 2   eid            328905 non-null  int64         
 3   event_name     328905 non-null  str           
 4   platform       328905 non-null  str           
 5   user_agent     328905 non-null  str           
 6   item_id        214308 non-null  float64       
 7   item_category  295985 non-null  str           
 8   item_location  305112 non-null  str           
 9   seller_type    195052 non-null  str           
 10  search_query   100402 non-null  str           
 11  search_page    100402 non-null  float64       
 12  pointer_x      108540 non-null  float64       
 13  pointer_y      108540 non-null  float64       
dtypes: datetime64[us](1), float64(4), int64(1), str(8)
memory usage

In [48]:
events['platform'] = events['platform'].str.lower().astype('category')
events['item_category'] =  events['item_category'].fillna('unknown').astype('category')
events['item_location'] = events['item_location'].fillna('unknown').astype('category')
events['seller_type'] = events['seller_type'].fillna('unknown').astype('category')
events['event_name'] = events['event_name'].astype('category')
q = events['search_query'].fillna('').astype('string')
events['query_entropy']   = q.apply(calculate_entropy)

## Осмотреться

Прежде чем считать агрегаты, стоит посмотреть на данные: какие типы событий бывают, что
лежит в `platform` и `user_agent`, где пропуски, всё ли уникально.

In [49]:
print(events.event_name.value_counts(), '\n')
print(events.platform.value_counts(), '\n')
print('пропуски по колонкам:')
print(events.isna().mean().round(3))

event_name
item_view               120817
search_results_view     100402
photo_swipe              36517
favorite_add             19049
seller_page_view         17403
contact_phone_show       11316
captcha_shown             7928
login                     6267
contact_chat_open         6125
contact_message_sent      3081
Name: count, dtype: int64 

platform
web        138792
android    126875
desktop     46866
ios         12269
iphone       4103
Name: count, dtype: int64 

пропуски по колонкам:
cookie_id        0.000
event_ts         0.000
eid              0.000
event_name       0.000
platform         0.000
user_agent       0.000
item_id          0.348
item_category    0.000
item_location    0.000
seller_type      0.000
search_query     0.695
search_page      0.695
pointer_x        0.670
pointer_y        0.670
query_entropy    0.000
dtype: float64


## Окно наблюдения

Признаки считаем только по событиям внутри окна: `window_start_ts <= event_ts < window_end_ts`.
Это требование из условия, а не рекомендация.

In [50]:
def events_in_window(events, meta):
    ev = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
    return ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]

ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)
print(len(ev_tr), len(ev_te))

198436 89690


In [51]:
def extract_all_features(ev: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    meta = meta.copy()
    meta['cookie_age_days'] = (meta['window_start_ts'] - meta['cookie_created_at']).dt.total_seconds() / 86400.0

    ev = ev.sort_values(['cookie_id', 'event_ts'])
    ev['time_diff'] = ev.groupby('cookie_id')['event_ts'].diff().dt.total_seconds()

    agg_spec = {
        'eid': 'count',
        'item_id': 'nunique',
        'item_category': 'nunique',
        'item_location': 'nunique',
        'search_query': 'nunique',
        'search_page': ['max', 'mean'],
        'entropy': ['mean', 'min', 'max'],
        'is_known_tool': 'max',
        'is_headless': 'max',
        'has_url': 'max',
        'starts_with_mozilla': 'mean',
        'is_app_header': 'max',
        'ua_device_conflict': 'max',
        'ua_length': ['mean', 'std'],
        'digit_ratio': 'mean',
        'chrome_major_version': ['min', 'max', 'nunique'],
        'time_diff': ['mean', 'std', 'min', 'median'],
        'pointer_x': lambda x: x.notna().mean(),
        'hour_sin_1': 'mean',
        'hour_cos_1': 'mean',
        'hour_sin_2': 'mean',
        'hour_cos_2': 'mean',
        'hour_distance_to_23': ['mean', 'min', 'std'],
        'is_night': 'mean',
        'is_evening': 'mean',
        'is_late_evening': 'mean',
    }

    features = ev.groupby('cookie_id').agg(agg_spec)
    features.columns = ['_'.join(c).strip('_') for c in features.columns]

    event_freq = pd.crosstab(ev['cookie_id'], ev['event_name'], normalize='index')
    event_freq.columns = [f'freq_{c}' for c in event_freq.columns]

    # Полный профиль активности по часам и разности этого профиля с лагами 1–3 часа.
    hour_profile = pd.crosstab(ev['cookie_id'], ev['event_hour'], normalize='index')
    hour_profile = hour_profile.reindex(columns=range(24), fill_value=0)
    hour_profile.columns = [f'hour_share_{h:02d}' for h in hour_profile.columns]
    hour_values = hour_profile.to_numpy()
    hour_lags = {}
    for lag in (1, 2, 3):
        # Разность с предыдущим часом; индекс часов циклический.
        diff = hour_values - np.roll(hour_values, lag, axis=1)
        for h in range(24):
            hour_lags[f'hour_diff_lag{lag}_{h:02d}'] = diff[:, h]
    hour_lags = pd.DataFrame(hour_lags, index=hour_profile.index)
    hour_lag_summary = pd.DataFrame(index=hour_profile.index)
    for lag in (1, 2, 3):
        diff = hour_values - np.roll(hour_values, lag, axis=1)
        hour_lag_summary[f'hour_diff_lag{lag}_mean_abs'] = np.abs(diff).mean(axis=1)
        hour_lag_summary[f'hour_diff_lag{lag}_max_abs'] = np.abs(diff).max(axis=1)

    hour_profile = hour_profile.join(hour_lags).join(hour_lag_summary)

    # Строковые представления для CatBoost: mode-категории и текстовые поля.
    def mode_or_missing(s):
        s = s.astype('string').fillna('__MISSING__')
        mode = s.mode()
        return str(mode.iloc[0]) if len(mode) else '__MISSING__'

    categorical_agg = ev.groupby('cookie_id').agg({
        'platform': mode_or_missing,
        'event_name': mode_or_missing,
        'item_category': mode_or_missing,
        'item_location': mode_or_missing,
        'seller_type': mode_or_missing,
        'user_agent': mode_or_missing,
    }).rename(columns={
        'platform': 'cat_platform_mode',
        'event_name': 'cat_event_name_mode',
        'item_category': 'cat_item_category_mode',
        'item_location': 'cat_item_location_mode',
        'seller_type': 'cat_seller_type_mode',
        'user_agent': 'text_user_agent',
    })

    def query_text(s):
        values = s.dropna().astype(str).drop_duplicates().head(100)
        return ' '.join(values) if len(values) else '__EMPTY_QUERY__'

    query_agg = ev.groupby('cookie_id')['search_query'].agg(query_text).rename('text_search_queries')

    df_out = meta[['cookie_id', 'cookie_age_days']].merge(features.reset_index(), on='cookie_id', how='left')
    df_out = df_out.merge(event_freq.reset_index(), on='cookie_id', how='left')
    df_out = df_out.merge(hour_profile.reset_index(), on='cookie_id', how='left')
    df_out = df_out.merge(categorical_agg.reset_index(), on='cookie_id', how='left')
    df_out = df_out.merge(query_agg.reset_index(), on='cookie_id', how='left')

    cat_text_cols = [
        'cat_platform_mode', 'cat_event_name_mode', 'cat_item_category_mode',
        'cat_item_location_mode', 'cat_seller_type_mode', 'text_user_agent',
        'text_search_queries'
    ]
    df_out[cat_text_cols] = df_out[cat_text_cols].fillna('__MISSING__').astype(str)
    numeric_cols = [c for c in df_out.columns if c not in cat_text_cols + ['cookie_id']]
    df_out[numeric_cols] = df_out[numeric_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    return df_out

In [52]:
add_ua_features(events)
add_temporal_features(events)

ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)

Xtr = extract_all_features(ev_tr, train)
Xte = extract_all_features(ev_te, test)

cat_feature_cols = [c for c in Xtr.columns if c.startswith('cat_')]
text_feature_cols = ['text_user_agent', 'text_search_queries']
catboost_feature_cols = [c for c in Xtr.columns if c != 'cookie_id']
feature_cols = [c for c in catboost_feature_cols if c not in cat_feature_cols + text_feature_cols]
ytr = train.target.values

print(f"Числовых признаков для LightGBM: {len(feature_cols)}")
print(f"Категориальных признаков для CatBoost: {len(cat_feature_cols)}")
print(f"Текстовых признаков для CatBoost: {len(text_feature_cols)}")

# Проверяем, что новые агрегаты безопасны для LightGBM.
numeric_Xtr = Xtr[feature_cols].select_dtypes(include=[np.number])
assert not numeric_Xtr.isna().any().any(), 'В Xtr остались NaN'
assert np.isfinite(numeric_Xtr.to_numpy()).all(), 'В Xtr есть inf/-inf'

Числовых признаков для LightGBM: 149
Категориальных признаков для CatBoost: 5
Текстовых признаков для CatBoost: 2


In [53]:
# Аудит входов моделей: проверяем, что train/test имеют одинаковый набор признаков.
assert set(Xtr.columns) == set(Xte.columns), 'Набор колонок Xtr и Xte различается'
assert set(catboost_feature_cols).issubset(Xtr.columns)
assert set(feature_cols).isdisjoint(set(cat_feature_cols + text_feature_cols))
print('Всего признаков в Xtr/Xte:', len(Xtr.columns) - 1)
print('LightGBM получает:', len(feature_cols), 'числовых признаков')
print('CatBoost получает:', len(catboost_feature_cols), 'признаков =', len(cat_feature_cols), 'cat +', len(text_feature_cols), 'text +', len(feature_cols), 'numeric')
print('Неиспользуемые моделью колонки:', sorted(set(Xtr.columns) - {'cookie_id'} - set(catboost_feature_cols)))

Всего признаков в Xtr/Xte: 156
LightGBM получает: 149 числовых признаков
CatBoost получает: 156 признаков = 5 cat + 2 text + 149 numeric
Неиспользуемые моделью колонки: []


## Валидация

Тест лежит **позже** трейна по времени, поэтому и валидацию честно делать по времени, а не
случайным сплитом.

Метрику берём из `metric.py` — это ровно тот код, которым считает проверяющая система.
Своя реализация почти наверняка разойдётся с официальной на одинаковых `score`:
их нельзя разделять, группа равных значений отмечается целиком.

In [54]:
import lightgbm as lgb
from metric import precision_at_recall

is_valid = train.window_start_ts.ge('2026-04-17').values

X_train_fold, y_train_fold = Xtr.loc[~is_valid, feature_cols], ytr[~is_valid]
X_val_fold, y_val_fold = Xtr.loc[is_valid, feature_cols], ytr[is_valid]

model = lgb.LGBMClassifier(
    n_estimators=400,
    learning_rate=0.03,
    max_depth=7,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    force_col_wise=True,
    n_jobs=-1,
)

model.fit(X_train_fold, y_train_fold)

val_preds = model.predict_proba(X_val_fold)[:, 1]
print('P@R0.7 на валидации:', round(precision_at_recall(y_val_fold, val_preds), 4))
print('Константа:', round(y_val_fold.mean(), 4))

P@R0.7 на валидации: 0.5022
Константа: 0.082


## CatBoost: категории и текстовые признаки

В LightGBM выше используются только числовые агрегаты. CatBoost получает mode-категории как `cat_features`, а User-Agent и объединённые поисковые запросы — как `text_features`.

In [55]:
cat_indices = [catboost_feature_cols.index(c) for c in cat_feature_cols]
text_indices = [catboost_feature_cols.index(c) for c in text_feature_cols]

X_train_cb = Xtr.loc[~is_valid, catboost_feature_cols].copy()
X_val_cb = Xtr.loc[is_valid, catboost_feature_cols].copy()
X_train_cb[cat_feature_cols + text_feature_cols] = X_train_cb[cat_feature_cols + text_feature_cols].fillna('__MISSING__').astype(str)
X_val_cb[cat_feature_cols + text_feature_cols] = X_val_cb[cat_feature_cols + text_feature_cols].fillna('__MISSING__').astype(str)

cat_model = CatBoostClassifier(
    iterations=2000,
    depth=8,
    learning_rate=0.01,
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=42,
    l2_leaf_reg=3,
    thread_count=-1,
    allow_writing_files=False,
    verbose=100
)

cat_model.fit(
    X_train_cb, y_train_fold,
    cat_features=cat_indices,
    text_features=text_indices,
    eval_set=(X_val_cb, y_val_fold),
    use_best_model=True,
    early_stopping_rounds=150,
    verbose=100
)
cat_val_preds = cat_model.predict_proba(X_val_cb)[:, 1]
print('CatBoost best_iteration:', cat_model.get_best_iteration())
print('CatBoost P@R0.7:', round(precision_at_recall(y_val_fold, cat_val_preds), 4))

0:	test: 0.6241136	best: 0.6241136 (0)	total: 139ms	remaining: 4m 37s
100:	test: 0.8533466	best: 0.8533466 (100)	total: 7.77s	remaining: 2m 26s
200:	test: 0.8723304	best: 0.8723304 (200)	total: 15.3s	remaining: 2m 16s
300:	test: 0.8769263	best: 0.8770345 (299)	total: 22.9s	remaining: 2m 9s
400:	test: 0.8778371	best: 0.8778371 (400)	total: 30.3s	remaining: 2m
500:	test: 0.8790375	best: 0.8790375 (499)	total: 37.6s	remaining: 1m 52s
600:	test: 0.8796413	best: 0.8797215 (598)	total: 45s	remaining: 1m 44s
700:	test: 0.8793412	best: 0.8797215 (598)	total: 52.3s	remaining: 1m 36s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.8797215243
bestIteration = 598

Shrink model to first 599 iterations.
CatBoost best_iteration: 598
CatBoost P@R0.7: 0.459


In [56]:
# Подбираем вес ансамбля только на временной валидации.
def rank01(values):
    return pd.Series(values).rank(method='average', pct=True).to_numpy()

blend_results = []
for cat_weight in np.linspace(0, 1, 11):
    raw_score = cat_weight * cat_val_preds + (1 - cat_weight) * val_preds
    rank_score = cat_weight * rank01(cat_val_preds) + (1 - cat_weight) * rank01(val_preds)
    blend_results.append({
        'cat_weight': cat_weight,
        'raw_p_at_r70': precision_at_recall(y_val_fold, raw_score),
        'rank_p_at_r70': precision_at_recall(y_val_fold, rank_score),
    })

blend_results = pd.DataFrame(blend_results)
display(blend_results.round(4))
best_raw = blend_results.loc[blend_results.raw_p_at_r70.idxmax()]
best_rank = blend_results.loc[blend_results.rank_p_at_r70.idxmax()]
if best_rank.rank_p_at_r70 > best_raw.raw_p_at_r70:
    ensemble_mode = 'rank'
    ensemble_cat_weight = float(best_rank.cat_weight)
    ensemble_val_score = float(best_rank.rank_p_at_r70)
else:
    ensemble_mode = 'raw'
    ensemble_cat_weight = float(best_raw.cat_weight)
    ensemble_val_score = float(best_raw.raw_p_at_r70)
print(f'Лучший режим: {ensemble_mode}, вес CatBoost: {ensemble_cat_weight:.1f}, P@R0.7: {ensemble_val_score:.4f}')

,cat_weight,raw_p_at_r70,rank_p_at_r70
0,0.0,0.5022,0.5022
1,0.1,0.5090,0.4786
2,0.2,0.5114,0.5161
3,0.3,0.5308,0.5209
4,0.4,0.5161,0.4978
5,0.5,0.4956,0.4746
6,0.6,0.4891,0.4647
7,0.7,0.4686,0.4650
8,0.8,0.4746,0.4809
9,0.9,0.4870,0.4726


Лучший режим: raw, вес CatBoost: 0.3, P@R0.7: 0.5308


## Сабмит

In [57]:
# Обучаем обе модели на всём train и смешиваем их предсказания.
model.fit(Xtr[feature_cols], ytr)
lgb_test_preds = model.predict_proba(Xte[feature_cols])[:, 1]

cat_iterations = max(1, cat_model.get_best_iteration() + 1)
X_full_cb = Xtr[catboost_feature_cols].copy()
X_test_cb = Xte[catboost_feature_cols].copy()
X_full_cb[cat_feature_cols + text_feature_cols] = X_full_cb[cat_feature_cols + text_feature_cols].fillna('__MISSING__').astype(str)
X_test_cb[cat_feature_cols + text_feature_cols] = X_test_cb[cat_feature_cols + text_feature_cols].fillna('__MISSING__').astype(str)

cat_model_full = CatBoostClassifier(
    iterations=cat_iterations,
    depth=8,
    learning_rate=0.01,
    loss_function='Logloss',
    random_seed=42,
    l2_leaf_reg=3,
    thread_count=-1,
    allow_writing_files=False,
    verbose=100
)
cat_model_full.fit(X_full_cb, ytr, cat_features=cat_indices, text_features=text_indices, verbose=100)
cat_test_preds = cat_model_full.predict_proba(X_test_cb)[:, 1]

if ensemble_mode == 'rank':
    ensemble_score = ensemble_cat_weight * rank01(cat_test_preds) + (1 - ensemble_cat_weight) * rank01(lgb_test_preds)
else:
    ensemble_score = ensemble_cat_weight * cat_test_preds + (1 - ensemble_cat_weight) * lgb_test_preds

sub = pd.DataFrame({'cookie_id': Xte.cookie_id, 'score': ensemble_score})
assert len(sub) == len(test) and sub.score.between(0, 1).all()
sub.to_csv('submission.csv', index=False)
print(f'Финальный ансамбль: {ensemble_mode}, CatBoost weight={ensemble_cat_weight:.1f}, iterations={cat_iterations}')
sub.head()

0:	learn: 0.6808190	total: 83.1ms	remaining: 49.7s
100:	learn: 0.2418787	total: 7.96s	remaining: 39.2s
200:	learn: 0.1742522	total: 15.6s	remaining: 30.9s
300:	learn: 0.1530808	total: 23.2s	remaining: 23s
400:	learn: 0.1444804	total: 30.8s	remaining: 15.2s
500:	learn: 0.1388049	total: 38.2s	remaining: 7.47s
598:	learn: 0.1349718	total: 45.3s	remaining: 0us
Финальный ансамбль: raw, CatBoost weight=0.3, iterations=599


,cookie_id,score
0,ck_315fb710a0e371e7,0.005604
1,ck_a76ee3b3e3e522fd,0.074967
2,ck_94c9a4d382689e82,0.015465
3,ck_8eaf9509ad9462a0,0.018802
4,ck_9a88a5a989cb5bc6,0.008399


## Куда копать дальше

Подсказок по конкретным признакам не будет — это и есть содержание задания. Несколько
вопросов, которые стоит себе задать:

* чем поток событий робота отличается от потока событий человека, если смотреть не на
  количество, а на **моменты времени**;
* что полезного лежит в строке `user_agent` и почему её нельзя брать как есть;
* насколько разнообразно то, что смотрит кука: объявления, категории, запросы, страницы выдачи;
* всё ли в порядке с самим файлом событий — порядок строк, дубликаты, пропуски;
* какие признаки бесполезны, потому что описывают технические характеристики, а не поведение.